In [1]:
from playgrounds.testing_regression import intrinsic_params
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append("/home/geraldine/Documents/Research/Projects/BayesGPT/")
print(sys.path)

['/snap/pycharm-professional/570/plugins/python-ce/helpers/jupyter_debug', '/snap/pycharm-professional/570/plugins/python-ce/helpers/pydev', '/home/geraldine/Documents/Research/Projects/BayesGPT', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/geraldine/Documents/Research/Projects/BayesGPT/.venv/lib/python3.12/site-packages', '/home/geraldine/Documents/Research/Projects/BayesGPT/']


In [5]:
import numpy as np

In [6]:
from bayesgpt.simulators.context_manager import ContextManager

In [14]:
context_manager = ContextManager()

In [15]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau"]

In [16]:
design_config = context_manager.build_design_config(
    intrinsic_params=intrinsic_params,
    regressed_params=["v", "a"],
    num_regressors=3,
    keep_intercept=True,
    add_interaction=True
)

In [17]:
for k, v in design_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['v', 'a']
u_2 ['v', 'a']
u_3 ['v', 'a']
u_1:u_2 ['v', 'a']
u_1:u_3 ['v', 'a']
u_2:u_3 ['v', 'a']


In [18]:
random_config = context_manager.build_random_design_config(
    intrinsic_params=intrinsic_params,
    num_regressors=3,
    free_intrinsics=intrinsic_params,
    fixed_intrinsics=[],
    keep_intercept=True,
    free_prob=0.5,
    add_interaction=True
)

In [19]:
for k, v in random_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['a', 's_v', 's_tau']
u_2 ['tau', 's_v']
u_3 ['v', 'a', 'tau', 's_tau']
u_1:u_2 ['s_v']
u_1:u_3 ['a', 's_tau']
u_2:u_3 ['tau']


In [20]:
X = context_manager.build_design_matrix(random_config, num_obs=100, keep_intercept=True, max_num_categories=4)

In [22]:
X.shape

(100, 19)

In [23]:
block_width = 3
keys = [k for k in random_config.keys() if k != "1"]

In [24]:
start = {k: i * block_width for i, k in enumerate(keys)}
for k in keys:
    if ":" in k:
        a, b = k.split(":")
        ok = np.allclose(X[:, start[k]], X[:, start[a]] * X[:, start[b]])
        print(k, "OK" if ok else "FAIL")

u_1:u_2 OK
u_1:u_3 OK
u_2:u_3 OK
